# CLIP on medical images (ROCO) — walkthrough

This notebook uses the `clip_roco` package from this repository to:
1. compute text–image similarity on natural and radiology images,
2. compare CLIP architectures on medical retrieval,
3. run zero-shot classification,
4. evaluate a CLIP model fine-tuned on ROCO.

Fine-tuning itself is launched with `scripts/finetune.py`.

## Setup (Colab)
Skip this cell if you run the notebook locally after `pip install -e .`.

In [ ]:
import os

if "COLAB_GPU" in os.environ or os.getcwd() == "/content":
    !git clone https://github.com/<your-username>/clip-medical-roco.git
    %cd clip-medical-roco
    !pip install -q -r requirements.txt
    !pip install -q -e .
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

from clip_roco.data import load_roco_samples, load_skimage_samples, parse_roco_captions
from clip_roco.metrics import compare_models, evaluate_similarity
from clip_roco.model import describe_model, get_device, list_models, load_clip
from clip_roco.samples import MEDICAL_DESCRIPTIONS, NATURAL_DESCRIPTIONS
from clip_roco.similarity import cosine_similarity_matrix
from clip_roco.visualization import plot_image_grid, plot_similarity_matrix, plot_zero_shot_predictions
from clip_roco.zero_shot import zero_shot_predict

ROCO_TRAIN_IMAGES = "data/roco/train/radiology/images"
ROCO_VAL_IMAGES = "data/roco/val/radiology/images"
ROCO_VAL_CAPTIONS = "data/roco/val/radiology/captions.txt"
CHECKPOINT = "outputs/checkpoints/vit-b16-roco/best_checkpoint.pth"

device = get_device()
device

## 1. Load CLIP

In [ ]:
print(list_models())
model, preprocess = load_clip("ViT-B/16", device=device)
describe_model(model)

## 2. Similarity on natural images

In [ ]:
nat_images, nat_texts = load_skimage_samples(NATURAL_DESCRIPTIONS)
plot_image_grid(nat_images, nat_texts)
nat_sim = cosine_similarity_matrix(model, preprocess, nat_images, nat_texts, device)
plot_similarity_matrix(nat_sim, nat_images, nat_texts, title="Cosine similarity (natural)")
evaluate_similarity(nat_sim)

## 3. Similarity on radiology images (pretrained CLIP)

In [ ]:
med_images, med_texts = load_roco_samples(ROCO_TRAIN_IMAGES, MEDICAL_DESCRIPTIONS)
med_sim = cosine_similarity_matrix(model, preprocess, med_images, med_texts, device)
plot_similarity_matrix(med_sim, med_images, med_texts, title="Cosine similarity (medical, pretrained)")
evaluate_similarity(med_sim)

## 4. Zero-shot classification on CIFAR-100 labels

In [ ]:
from torchvision.datasets import CIFAR100

cifar100 = CIFAR100("data/torchvision", train=False, download=True)
top_probs, top_labels = zero_shot_predict(model, preprocess, nat_images, cifar100.classes, device)
plot_zero_shot_predictions(nat_images, top_probs, top_labels, cifar100.classes)

## 5. Comparing CLIP architectures on medical retrieval

In [ ]:
captions = parse_roco_captions(ROCO_VAL_CAPTIONS)
val_images, val_texts = load_roco_samples(ROCO_VAL_IMAGES, captions, limit=15)

similarities = {}
for name in list_models():
    m, p = load_clip(name, device=device)
    similarities[name] = cosine_similarity_matrix(m, p, val_images, val_texts, device)
    del m

compare_models(similarities)

## 6. After fine-tuning on ROCO
Train first with `python scripts/finetune.py`.

In [ ]:
ft_model, ft_preprocess = load_clip("ViT-B/16", device=device, checkpoint=CHECKPOINT)
ft_sim = cosine_similarity_matrix(ft_model, ft_preprocess, med_images, med_texts, device)
plot_similarity_matrix(ft_sim, med_images, med_texts, title="Cosine similarity (medical, fine-tuned)")
compare_models({"pretrained": med_sim, "fine-tuned": ft_sim})